# 02. 신규 전처리 — 화성시 불법주정차 단속현황 (xlsx)
**에네레기파 (임단군, 정예민, 안유경) | 수원대학교 데이터과학부**

**입력:** `data/원본/화성시 불법주정차 단속현황 자료.xlsx` (2023년, 217,682건)

**출력 파일 4종** → `data/전처리_신규/`

| 파일 | 설명 | 용도 |
|------|------|------|
| `행정동별_실단속위험도.csv` | 실단속 기반 위험도 재보정 | 지도·위험도 |
| `행정동별_위반유형집계.csv` | 위반유형 + 통행불편지수 | 위반유형 분석 |
| `시간대별_단속집계.csv` | 요일×시간대 피벗 | 히트맵 |
| `CCTV공백지수.csv` | 민원비율 vs CCTV비율 | CCTV 공백 지도 |


## 1. 라이브러리 임포트

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')
print('라이브러리 로드 완료')

## 2. 데이터 로드

xlsx 파일은 3개 부서 시트로 구성됨:
- **본청 주차교통과** (83,196건)
- **동부출장소 교통건설과** (22,807건)
- **동탄출장소 교통건설과** (111,679건) — 전체의 51%


In [ ]:
XLSX_PATH = '../data/원본/화성시 불법주정차 단속현황 자료.xlsx'
RISK_PATH = '../data/전처리_기존/행정동별위험도점수.csv'
OUT_DIR   = '../data/전처리_신규/'

sheets = pd.read_excel(XLSX_PATH, sheet_name=None)
df = pd.concat(sheets.values(), ignore_index=True)

df['단속일시'] = pd.to_datetime(df['단속일시'])
df['연도'] = df['단속일시'].dt.year
df['월']   = df['단속일시'].dt.month
df['요일']  = df['단속일시'].dt.dayofweek
df['시간']  = df['단속일시'].dt.hour

print(f'전체 레코드 수: {len(df):,}건')
print(f'기간: {df["단속일시"].min()} ~ {df["단속일시"].max()}')
display(df.head(3))

## 3. 행정동 파싱

`단속장소` 첫 번째 토큰에서 읍/동/면으로 끝나는 행정단위 추출.


In [ ]:
DONG_ALIAS = {
    "향남": "향남읍", "남양": "남양읍", "봉담": "봉담읍", "우정": "우정읍",
    "마도": "마도면", "송산": "송산면", "서신": "서신면", "팔탄": "팔탄면",
    "양감": "양감면", "정남": "정남면", "비봉": "비봉면", "장안": "장안면",
}

def extract_dong(loc):
    if pd.isna(loc):
        return None
    tokens = loc.split()
    for t in tokens[:3]:
        t_clean = re.sub(r"^[^가-힣]+", "", t)
        if re.search(r"(동|읍|면)$", t_clean):
            return t_clean
    first = re.sub(r"^[^가-힣]+", "", tokens[0]) if tokens else ""
    return DONG_ALIAS.get(first, None)

df["행정동"] = df["단속장소"].apply(extract_dong)
miss = df["행정동"].isna().sum()
print(f"추출 완료 | 미추출: {miss}건 ({miss/len(df)*100:.1f}%)")
print("상위 15 행정동:")
print(df["행정동"].value_counts().head(15).to_string())


## 4. 위반유형 파싱 + 통행불편 가중치

`단속장소` 괄호 안 텍스트(숫자 제외)를 위반유형으로 분류.

| 카테고리 | 통행불편 가중치 |
|---------|---------------|
| 횡단보도 / 소화전 / 버스정류소 | 2.0 |
| 인도 / 도로가장자리 | 1.5 |
| 모퉁이·교차로 | 1.2 |
| 기타 | 1.0 |


In [ ]:
df['위반유형_raw'] = df['단속장소'].str.extract(r'\(([^)]+)\)')
df.loc[df['위반유형_raw'].str.match(r'^\d+$', na=False), '위반유형_raw'] = None

VTYPE_MAP = {
    '횡단보도':      ('횡단보도',      2.0),
    '소화전':        ('소화전',        2.0),
    '버스정류소':    ('버스정류',      2.0),
    '인도':          ('인도',          1.5),
    '도로가장자리':  ('도로가장자리',  1.5),
    '모퉁이·교차로': ('모퉁이|교차로', 1.2),
}

def classify_vtype(raw):
    if pd.isna(raw):
        return ('기타', 1.0)
    for label, (pattern, weight) in VTYPE_MAP.items():
        if re.search(pattern, raw):
            return (label, weight)
    return ('기타', 1.0)

df[['위반유형', '통행불편가중치']] = df['위반유형_raw'].apply(
    lambda x: pd.Series(classify_vtype(x))
)

print('위반유형 분포:')
print(df['위반유형'].value_counts().to_string())

## 5. 시간대 구간 피처

In [ ]:
def time_band(h):
    if 7 <= h < 12:  return "오전피크"
    if 12 <= h < 15: return "점심"
    if 15 <= h < 19: return "오후피크"
    if 19 <= h < 22: return "저녁"
    return "야간"

df["시간대구간"] = df["시간"].apply(time_band)
df["평일여부"]   = df["요일"].apply(lambda d: "평일" if d < 5 else "주말")
df["요일명"]     = df["요일"].map({0:"월",1:"화",2:"수",3:"목",4:"금",5:"토",6:"일"})

# 행정동 파싱 성공 레코드만 필터링
df_valid = df[df["행정동"].notna()].copy()
print(f"유효 행정동 레코드: {len(df_valid):,}건 | 행정동 수: {df_valid["행정동"].nunique()}")

print("시간대구간 분포:")
print(df["시간대구간"].value_counts().to_string())


## 6. [OUTPUT 1] 행정동별_실단속위험도.csv

실단속 데이터 기반 위험도 재보정 (상권 데이터 의존도 제거).

**재보정위험도점수** = 실단속비율_점수 × **0.65** + 민원비율_점수 × **0.35**

| 구성 요소 | 가중치 | 의미 |
|---------|--------|------|
| 실단속비율_점수 | 0.65 | 해당 동의 실제 단속 집중도 (전체 대비 비율 → MinMax 0~10) |
| 민원비율_점수 | 0.35 | 민원 신고 집중도 (전체 대비 비율 → MinMax 0~10) |

> 상권위험도는 별도 데이터 기반이라 교차 오염 방지를 위해 가중치에서 제외. CSV에는 보조 컬럼으로만 저장.

In [ ]:
dong_total = df_valid.groupby('행정동').size().reset_index(name='실단속건수')
dong_민원  = df[df['단속구분'] == '민원(공익제보)'].groupby('행정동').size().reset_index(name='민원건수')
dong_cctv  = df[df['단속구분'].str.contains('CCTV', na=False)].groupby('행정동').size().reset_index(name='CCTV건수')

dong = dong_total.merge(dong_민원, on='행정동', how='left')
dong = dong.merge(dong_cctv,  on='행정동', how='left').fillna(0)

dong['민원비율']          = (dong['민원건수'] / dong['실단속건수']).round(4)
dong['CCTV비율']         = (dong['CCTV건수']  / dong['실단속건수']).round(4)
dong['실단속비율_정규화'] = (dong['실단속건수'] / dong['실단속건수'].sum()).round(6)

# 상권 데이터는 보조 참고용으로만 저장 (위험도 가중치에는 미포함)
risk_old = pd.read_csv(RISK_PATH, encoding='utf-8-sig')[['행정동명', '위험도점수']]
dong = dong.merge(risk_old, left_on='행정동', right_on='행정동명', how='left').drop(columns=['행정동명'])
dong['위험도점수'] = dong['위험도점수'].fillna(0)

scaler = MinMaxScaler(feature_range=(0, 10))
dong['상권위험도_정규화'] = scaler.fit_transform(dong[['위험도점수']]).round(2)  # 보조용
dong['실단속비율_점수']   = scaler.fit_transform(dong[['실단속비율_정규화']]).round(2)
dong['민원비율_점수']     = scaler.fit_transform(dong[['민원비율']]).round(2)

# 재보정위험도 = 실단속비율(0.65) + 민원비율(0.35)  — 상권 제외
dong['재보정위험도점수'] = (
    dong['실단속비율_점수'] * 0.65 +
    dong['민원비율_점수']   * 0.35
).round(2)

dong['위험등급'] = pd.cut(
    dong['재보정위험도점수'],
    bins=[0, 3, 6, 10],
    labels=['저위험', '중위험', '고위험'],
    include_lowest=True
)
dong = dong.sort_values('재보정위험도점수', ascending=False).reset_index(drop=True)

print('=== 행정동별 재보정 위험도 ===')
display(dong[['행정동', '실단속건수', '민원비율', 'CCTV비율', '재보정위험도점수', '위험등급']])

dong.to_csv(OUT_DIR + '행정동별_실단속위험도.csv', index=False, encoding='utf-8-sig')
print('저장 완료: 행정동별_실단속위험도.csv')

## 7. [OUTPUT 2] 행정동별_위반유형집계.csv

행정동 × 위반유형 피벗 + **통행불편지수** 계산.

**통행불편지수** = Σ(위반건수 × 가중치) / 전체건수


In [ ]:
vtype_pivot = (
    df.groupby(['행정동', '위반유형'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

WEIGHT = {
    '횡단보도': 2.0, '소화전': 2.0, '버스정류소': 2.0,
    '인도': 1.5, '도로가장자리': 1.5, '모퉁이·교차로': 1.2, '기타': 1.0
}

def calc_idx(row):
    total = sum(row.get(k, 0) * w for k, w in WEIGHT.items())
    cnt   = sum(row.get(k, 0) for k in WEIGHT)
    return round(total / cnt, 4) if cnt > 0 else 0

vtype_pivot['통행불편지수'] = vtype_pivot.apply(calc_idx, axis=1)
vtype_pivot = vtype_pivot.merge(
    dong[['행정동', '실단속건수', '재보정위험도점수', '위험등급']],
    on='행정동', how='left'
).sort_values('통행불편지수', ascending=False).reset_index(drop=True)

cols = ['행정동','실단속건수','통행불편지수','횡단보도','소화전','인도','버스정류소','기타']
cols = [c for c in cols if c in vtype_pivot.columns]
display(vtype_pivot[cols].head(10))

vtype_pivot.to_csv(OUT_DIR + '행정동별_위반유형집계.csv', index=False, encoding='utf-8-sig')
print('저장 완료: 행정동별_위반유형집계.csv')

## 8. [OUTPUT 3] 시간대별_단속집계.csv

요일(월~일) × 시간(0~23) 피벗 테이블 — 대시보드 히트맵 데이터.


In [ ]:
heat = df.groupby(["요일명","시간"]).size().reset_index(name="단속건수")
heat_pivot = heat.pivot(index="요일명", columns="시간", values="단속건수").fillna(0).astype(int)
heat_pivot = heat_pivot.reindex(["월","화","수","목","금","토","일"])

print("=== 요일×시간 히트맵 피벗 (6~21시) ===")
display(heat_pivot.iloc[:, 6:22])

heat.to_csv(OUT_DIR + "시간대별_단속집계.csv", index=False, encoding="utf-8-sig")
heat_pivot.reset_index().to_csv(OUT_DIR + "시간대별_단속피벗.csv", index=False, encoding="utf-8-sig")
print("저장 완료: 시간대별_단속집계.csv / 시간대별_단속피벗.csv")


## 9. [OUTPUT 4] CCTV공백지수.csv

**CCTV 공백 지수** = 민원비율 − 고정형CCTV비율

값이 높을수록: 주민 신고는 많은데 고정CCTV 인프라가 없는 지역 → 설치 우선순위.


In [ ]:
cctv_df = dong[['행정동','실단속건수','민원비율','CCTV비율','재보정위험도점수','위험등급']].copy()

fixed = df[df['단속구분'] == '고정형CCTV'].groupby('행정동').size().reset_index(name='고정CCTV건수')
cctv_df = cctv_df.merge(fixed, on='행정동', how='left').fillna({'고정CCTV건수': 0})
cctv_df['고정CCTV비율'] = (cctv_df['고정CCTV건수'] / cctv_df['실단속건수']).round(4)
cctv_df['CCTV공백지수'] = (cctv_df['민원비율'] - cctv_df['고정CCTV비율']).round(4)
cctv_df['CCTV설치우선순위'] = pd.cut(
    cctv_df['CCTV공백지수'], bins=[-1, 0, 0.1, 1],
    labels=['낮음','보통','높음'], include_lowest=True
)
cctv_df = cctv_df.sort_values('CCTV공백지수', ascending=False).reset_index(drop=True)

display(cctv_df[['행정동','민원비율','고정CCTV비율','CCTV공백지수','CCTV설치우선순위']])

cctv_df.to_csv(OUT_DIR + 'CCTV공백지수.csv', index=False, encoding='utf-8-sig')
print('저장 완료: CCTV공백지수.csv')

## 10. 생성 파일 최종 확인

In [ ]:
import os
print('=== data/전처리_신규/ 생성 파일 ===')
for f in sorted(os.listdir(OUT_DIR)):
    size = os.path.getsize(OUT_DIR + f)
    print(f'  {f:<45} {size:>10,} bytes')